# Generative AI Lab Assignment - 1
**Course:** MCA – Generative AI Lab  
**Title:** *Generating Handwritten Digits using Variational Autoencoder (VAE) and Generative Adversarial Network (GAN)*

---

## Problem Statement
Develop and implement **deep generative models**, namely **Variational Autoencoder (VAE)** and **Generative Adversarial Network (GAN)**, to learn the data distribution of handwritten digit images and generate new digit samples. The system visualizes both the **latent space** and **generated outputs**.

---

## Step 1: Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import os

# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Create directory to save generated images
os.makedirs('generated_images', exist_ok=True)
print('Setup complete!')

---
## Step 2: Dataset Preparation
Loading and normalizing the **MNIST** dataset (28×28 grayscale images).

In [ ]:
# ─── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE   = 128        # Number of samples per training batch
LATENT_DIM   = 2          # Latent space dimensions (2D for easy visualization)
VAE_EPOCHS   = 20         # Training epochs for VAE
GAN_EPOCHS   = 50         # Training epochs for GAN
NOISE_DIM    = 100         # Input noise dimension for GAN Generator
LR_VAE       = 1e-3       # Learning rate for VAE
LR_GAN       = 2e-4       # Learning rate for GAN
IMAGE_SIZE   = 28 * 28    # Flattened image size
# ───────────────────────────────────────────────────────────────────────────────

# Normalize images to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()  # Converts PIL image to tensor and scales to [0,1]
])

# Download and load MNIST training dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)

# DataLoader for batch processing
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print(f'Total training samples : {len(train_dataset)}')
print(f'Number of batches       : {len(train_loader)}')
print(f'Image shape             : 28 x 28 grayscale')

# ── Visualize original dataset samples ────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('Original MNIST Dataset Samples', fontsize=14, fontweight='bold')
sample_images, sample_labels = next(iter(train_loader))

for i, ax in enumerate(axes.flat):
    ax.imshow(sample_images[i].squeeze(), cmap='gray')
    ax.set_title(f'Label: {sample_labels[i].item()}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('generated_images/01_original_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Original samples saved.')

---
## Step 3: Variational Autoencoder (VAE) Implementation

### Architecture:
- **Encoder**: Compresses input image → mean (μ) and log-variance (log σ²)
- **Reparameterization Trick**: z = μ + σ * ε, where ε ~ N(0, I)
- **Decoder**: Reconstructs image from latent vector z

### Loss Function:
$$\mathcal{L}_{VAE} = \underbrace{\text{Reconstruction Loss}}_{\text{BCE}} + \underbrace{\text{KL Divergence}}_{\text{Regularization}}$$

In [ ]:
class VAE(nn.Module):
    """
    Variational Autoencoder (VAE) for MNIST digit generation.
    
    Architecture:
        Encoder : 784 → 512 → 256 → [μ, logσ²] (LATENT_DIM each)
        Decoder : LATENT_DIM → 256 → 512 → 784
    """
    def __init__(self, image_size=784, latent_dim=2):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim

        # ── Encoder ──────────────────────────────────────────────────────────
        self.encoder = nn.Sequential(
            nn.Linear(image_size, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU()
        )
        # Produces mean and log-variance for the latent distribution
        self.fc_mu     = nn.Linear(256, latent_dim)  # Mean vector
        self.fc_logvar = nn.Linear(256, latent_dim)  # Log-variance vector

        # ── Decoder ──────────────────────────────────────────────────────────
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, image_size),
            nn.Sigmoid()  # Output in [0, 1] to match normalized image pixels
        )

    def encode(self, x):
        """Encode input image to latent distribution parameters."""
        h      = self.encoder(x)
        mu     = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick: z = mu + std * epsilon
        Enables backpropagation through stochastic sampling.
        """
        if self.training:
            std = torch.exp(0.5 * logvar)      # σ = exp(0.5 * log σ²)
            eps = torch.randn_like(std)        # ε ~ N(0, I)
            return mu + eps * std
        return mu  # During inference, use mean directly

    def decode(self, z):
        """Decode latent vector back to image space."""
        return self.decoder(z)

    def forward(self, x):
        """Full forward pass: encode → reparameterize → decode."""
        mu, logvar = self.encode(x)
        z          = self.reparameterize(mu, logvar)
        recon_x    = self.decode(z)
        return recon_x, mu, logvar


def vae_loss(recon_x, x, mu, logvar):
    """
    VAE Loss = Reconstruction Loss (BCE) + KL Divergence
    
    - BCE  : Measures pixel-wise reconstruction accuracy
    - KLD  : Regularizes latent space toward N(0, I)
             KLD = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    """
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD


# Instantiate model and optimizer
vae_model     = VAE(image_size=IMAGE_SIZE, latent_dim=LATENT_DIM).to(device)
vae_optimizer = optim.Adam(vae_model.parameters(), lr=LR_VAE)

print('VAE Model Architecture:')
print(vae_model)
total_params = sum(p.numel() for p in vae_model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')

### Train the VAE Model

In [ ]:
def train_vae(model, optimizer, loader, epochs):
    """
    Training loop for the VAE.
    Returns list of epoch losses for plotting.
    """
    model.train()
    loss_history = []

    for epoch in range(1, epochs + 1):
        total_loss = 0

        for batch_images, _ in loader:
            # Flatten images from (B, 1, 28, 28) → (B, 784)
            x = batch_images.view(-1, IMAGE_SIZE).to(device)

            optimizer.zero_grad()                       # Clear gradients
            recon_x, mu, logvar = model(x)              # Forward pass
            loss = vae_loss(recon_x, x, mu, logvar)     # Compute loss
            loss.backward()                             # Backpropagation
            optimizer.step()                            # Update weights

            total_loss += loss.item()

        avg_loss = total_loss / len(loader.dataset)
        loss_history.append(avg_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch [{epoch:>3}/{epochs}]  |  VAE Loss: {avg_loss:.4f}')

    return loss_history


print('Training VAE...')
print('─' * 40)
vae_loss_history = train_vae(vae_model, vae_optimizer, train_loader, VAE_EPOCHS)
print('─' * 40)
print('VAE Training Complete!')

# ── Plot VAE Training Loss ─────────────────────────────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(range(1, VAE_EPOCHS + 1), vae_loss_history, color='steelblue', linewidth=2, marker='o', markersize=4)
plt.title('VAE Training Loss over Epochs', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss (BCE + KLD)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('generated_images/02_vae_training_loss.png', dpi=150)
plt.show()

### Visualize VAE Generated Images

In [ ]:
def generate_vae_images(model, num_images=16, latent_dim=2):
    """
    Generate new images by sampling random latent vectors
    from N(0, I) and passing through the VAE decoder.
    """
    model.eval()
    with torch.no_grad():
        # Sample random latent vectors
        z = torch.randn(num_images, latent_dim).to(device)
        generated = model.decode(z)                         # Decode to image space
        generated = generated.view(-1, 1, 28, 28).cpu()    # Reshape to image format
    return generated


vae_generated = generate_vae_images(vae_model, num_images=16, latent_dim=LATENT_DIM)

# ── Display VAE Generated Images ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('VAE Generated Digit Images (from random latent vectors)', fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(vae_generated[i].squeeze(), cmap='gray')
    ax.axis('off')

plt.tight_layout()
plt.savefig('generated_images/03_vae_generated_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('VAE generated images saved.')

### Visualize VAE Latent Space (2D Projection)

In [ ]:
def visualize_latent_space(model, loader, num_batches=10):
    """
    Encode real images into latent space and plot the 2D distribution.
    Colors represent digit classes (0-9).
    """
    model.eval()
    all_mu     = []
    all_labels = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            if i >= num_batches:
                break
            x          = images.view(-1, IMAGE_SIZE).to(device)
            mu, _      = model.encode(x)    # Get latent mean vectors
            all_mu.append(mu.cpu().numpy())
            all_labels.append(labels.numpy())

    all_mu     = np.concatenate(all_mu,     axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # ── Plot Latent Space Distribution ────────────────────────────────────────
    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(
        all_mu[:, 0], all_mu[:, 1],
        c=all_labels, cmap='tab10',
        alpha=0.7, s=12
    )
    cbar = plt.colorbar(scatter, ticks=range(10))
    cbar.set_label('Digit Class', rotation=270, labelpad=15)
    plt.title('VAE Latent Space (2D Projection)\nEach color = one digit class', fontsize=13, fontweight='bold')
    plt.xlabel('Latent Dimension 1 (z₁)')
    plt.ylabel('Latent Dimension 2 (z₂)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('generated_images/04_vae_latent_space.png', dpi=150)
    plt.show()
    print('Latent space visualization saved.')


visualize_latent_space(vae_model, train_loader, num_batches=15)

### Latent Space Traversal (Interpolation between digits)

In [ ]:
def latent_space_traversal(model, n=15, latent_dim=2):
    """
    Traverse the 2D latent space in a grid pattern to see
    how generated digits change across different latent values.
    """
    model.eval()
    figure  = np.zeros((28 * n, 28 * n))
    grid    = np.linspace(-3, 3, n)  # Traverse latent space from -3 to +3

    with torch.no_grad():
        for i, yi in enumerate(grid):
            for j, xi in enumerate(grid):
                z_sample = torch.tensor([[xi, yi]], dtype=torch.float32).to(device)
                x_decoded = model.decode(z_sample)
                digit     = x_decoded[0].cpu().numpy().reshape(28, 28)
                figure[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = digit

    plt.figure(figsize=(10, 10))
    plt.imshow(figure, cmap='gray')
    plt.title('VAE Latent Space Traversal\n(Each cell = decoded image at that (z₁, z₂) coordinate)',
              fontsize=12, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('generated_images/05_vae_latent_traversal.png', dpi=150)
    plt.show()
    print('Latent space traversal saved.')


latent_space_traversal(vae_model, n=15, latent_dim=LATENT_DIM)

---
## Step 4: Generative Adversarial Network (GAN) Implementation

### Architecture:
- **Generator (G)**: Converts random noise z → fake image
- **Discriminator (D)**: Classifies images as real or fake
- **Adversarial Training**: G and D compete against each other

### Loss Functions:
$$\mathcal{L}_D = -\mathbb{E}[\log D(x)] - \mathbb{E}[\log(1-D(G(z)))]$$
$$\mathcal{L}_G = -\mathbb{E}[\log D(G(z))]$$

In [ ]:
class Generator(nn.Module):
    """
    GAN Generator Network.
    Maps random noise vector z → fake image.
    
    Architecture: noise_dim → 256 → 512 → 1024 → 784
    """
    def __init__(self, noise_dim=100, image_size=784):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            # Layer 1
            nn.Linear(noise_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(256),

            # Layer 2
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(512),

            # Layer 3
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.BatchNorm1d(1024),

            # Output layer
            nn.Linear(1024, image_size),
            nn.Tanh()  # Output in [-1, 1] (images normalized to this range)
        )

    def forward(self, z):
        return self.model(z)


class Discriminator(nn.Module):
    """
    GAN Discriminator Network.
    Classifies input image as Real (1) or Fake (0).
    
    Architecture: 784 → 512 → 256 → 128 → 1
    """
    def __init__(self, image_size=784):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            # Layer 1
            nn.Linear(image_size, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            # Layer 2
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            # Layer 3
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2, inplace=True),

            # Output layer
            nn.Linear(128, 1),
            nn.Sigmoid()  # Output probability: 1 = real, 0 = fake
        )

    def forward(self, x):
        return self.model(x)


# Instantiate Generator and Discriminator
generator     = Generator(noise_dim=NOISE_DIM, image_size=IMAGE_SIZE).to(device)
discriminator = Discriminator(image_size=IMAGE_SIZE).to(device)

# Separate optimizers for Generator and Discriminator
g_optimizer = optim.Adam(generator.parameters(),     lr=LR_GAN, betas=(0.5, 0.999))
d_optimizer = optim.Adam(discriminator.parameters(), lr=LR_GAN, betas=(0.5, 0.999))

# Binary Cross Entropy loss for adversarial training
criterion = nn.BCELoss()

print('Generator Architecture:')
print(generator)
print('\nDiscriminator Architecture:')
print(discriminator)

### Train the GAN Model

In [ ]:
# GAN images must be normalized to [-1, 1] to match Generator's Tanh output
gan_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [0,1] → [-1,1]
])

gan_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=gan_transform, download=False
)
gan_loader = DataLoader(gan_dataset, batch_size=BATCH_SIZE, shuffle=True)


def train_gan(gen, disc, g_opt, d_opt, loader, epochs):
    """
    GAN Training Loop using alternating optimization:
      1. Train Discriminator to distinguish real vs fake
      2. Train Generator to fool the Discriminator
    """
    g_losses = []
    d_losses = []

    for epoch in range(1, epochs + 1):
        epoch_g_loss = 0
        epoch_d_loss = 0

        for real_images, _ in loader:
            batch_size = real_images.size(0)
            real_imgs  = real_images.view(batch_size, -1).to(device)  # Flatten

            # ── Labels for real and fake images ───────────────────────────────
            real_labels = torch.ones(batch_size, 1).to(device)   # 1 = real
            fake_labels = torch.zeros(batch_size, 1).to(device)  # 0 = fake

            # ══ Train Discriminator ══════════════════════════════════════════
            disc.train(); gen.eval()
            d_opt.zero_grad()

            # Loss on real images: D should predict 1
            real_preds   = disc(real_imgs)
            d_loss_real  = criterion(real_preds, real_labels)

            # Generate fake images from noise
            z            = torch.randn(batch_size, NOISE_DIM).to(device)
            fake_imgs    = gen(z).detach()  # detach: don't update generator here
            fake_preds   = disc(fake_imgs)
            d_loss_fake  = criterion(fake_preds, fake_labels)

            d_loss       = d_loss_real + d_loss_fake
            d_loss.backward()
            d_opt.step()

            # ══ Train Generator ══════════════════════════════════════════════
            gen.train(); disc.eval()
            g_opt.zero_grad()

            # Generator wants Discriminator to predict fake images as real (1)
            z          = torch.randn(batch_size, NOISE_DIM).to(device)
            fake_imgs  = gen(z)
            g_preds    = disc(fake_imgs)
            g_loss     = criterion(g_preds, real_labels)  # Fool D → target = 1

            g_loss.backward()
            g_opt.step()

            epoch_d_loss += d_loss.item()
            epoch_g_loss += g_loss.item()

        avg_d = epoch_d_loss / len(loader)
        avg_g = epoch_g_loss / len(loader)
        d_losses.append(avg_d)
        g_losses.append(avg_g)

        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch [{epoch:>3}/{epochs}]  |  D Loss: {avg_d:.4f}  |  G Loss: {avg_g:.4f}')

    return g_losses, d_losses


print('Training GAN...')
print('─' * 50)
g_loss_hist, d_loss_hist = train_gan(
    generator, discriminator,
    g_optimizer, d_optimizer,
    gan_loader, GAN_EPOCHS
)
print('─' * 50)
print('GAN Training Complete!')

# ── Plot GAN Training Losses ───────────────────────────────────────────────────
plt.figure(figsize=(9, 4))
plt.plot(range(1, GAN_EPOCHS + 1), g_loss_hist, label='Generator Loss',     color='tomato',    linewidth=2)
plt.plot(range(1, GAN_EPOCHS + 1), d_loss_hist, label='Discriminator Loss', color='steelblue', linewidth=2)
plt.title('GAN Training Loss over Epochs', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('generated_images/06_gan_training_loss.png', dpi=150)
plt.show()

### Visualize GAN Generated Images

In [ ]:
def generate_gan_images(gen, num_images=16, noise_dim=100):
    """
    Generate images from the trained GAN Generator.
    Samples random noise from N(0, I) and forward through G.
    """
    gen.eval()
    with torch.no_grad():
        z         = torch.randn(num_images, noise_dim).to(device)
        generated = gen(z)                               # Generate fake images
        generated = generated.view(-1, 1, 28, 28).cpu() # Reshape
        generated = (generated + 1) / 2                 # [-1,1] → [0,1] for display
    return generated


gan_generated = generate_gan_images(generator, num_images=16, noise_dim=NOISE_DIM)

# ── Display GAN Generated Images ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('GAN Generated Digit Images (from random noise vectors)', fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(gan_generated[i].squeeze(), cmap='gray')
    ax.axis('off')

plt.tight_layout()
plt.savefig('generated_images/07_gan_generated_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('GAN generated images saved.')

---
## Step 5: Comparative Analysis — VAE vs GAN

In [ ]:
# ── Side-by-side comparison: Original | VAE | GAN ─────────────────────────────
fig, axes = plt.subplots(3, 8, figsize=(15, 6))

row_titles = ['Original MNIST Samples', 'VAE Generated Samples', 'GAN Generated Samples']
image_sets = [sample_images[:8], vae_generated[:8], gan_generated[:8]]

for row_idx, (row_title, images) in enumerate(zip(row_titles, image_sets)):
    axes[row_idx][0].set_ylabel(row_title, fontsize=9, fontweight='bold', rotation=90, labelpad=5)
    for col_idx in range(8):
        ax = axes[row_idx][col_idx]
        ax.imshow(images[col_idx].squeeze(), cmap='gray')
        ax.axis('off')

plt.suptitle('VAE vs GAN: Side-by-Side Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('generated_images/08_comparison_vae_vs_gan.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparison plot saved.')

---
## Step 6: Analysis & Observations

### Qualitative Comparison

| Feature | VAE | GAN |
|---|---|---|
| **Image Sharpness** | Smoother / slightly blurred | Sharper and crisper |
| **Diversity** | Good diversity via latent sampling | Good diversity from noise |
| **Training Stability** | Stable (single loss function) | Can be unstable (mode collapse) |
| **Latent Space** | Structured, continuous & interpretable | No explicit latent space |
| **Interpolation** | Smooth interpolation between digits | Not directly possible |
| **Loss Function** | BCE + KL Divergence | Adversarial (minimax) |
| **Inference Speed** | Fast (encoder + decoder) | Fast (generator only) |

In [ ]:
# ── Analysis Bar Chart ─────────────────────────────────────────────────────────
criteria   = ['Image\nSharpness', 'Training\nStability', 'Latent Space\nInterpretability',
               'Generation\nDiversity', 'Speed']
vae_scores = [6, 9, 10, 7, 8]   # Scores out of 10 (qualitative)
gan_scores = [9, 6,  3, 8, 9]

x     = np.arange(len(criteria))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, vae_scores, width, label='VAE', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, gan_scores, width, label='GAN', color='darkorange', alpha=0.85)

ax.set_title('VAE vs GAN — Qualitative Performance Comparison', fontsize=13, fontweight='bold')
ax.set_ylabel('Score (out of 10)')
ax.set_xticks(x)
ax.set_xticklabels(criteria)
ax.set_ylim(0, 12)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('generated_images/09_vae_vs_gan_analysis.png', dpi=150)
plt.show()

---
## Conclusion

In this assignment, we successfully implemented and trained two powerful **deep generative models** — VAE and GAN — on the MNIST handwritten digits dataset.

### Key Observations:
1. **VAE** provides a structured, interpretable latent space. The 2D latent space traversal clearly shows smooth transitions between digit classes, confirming the model's ability to learn a meaningful data distribution.
2. **GAN** produces sharper and more realistic-looking digits. The adversarial training mechanism allows the Generator to progressively improve by competing against the Discriminator.
3. **VAE** is easier to train due to its well-defined loss function (BCE + KLD), while **GAN** training is more sensitive to hyperparameter tuning and can suffer from mode collapse.
4. Both models are capable of generating novel digit images that were never seen during training, demonstrating the power of **generative modeling**.

### Use Cases:
- **VAE**: Suitable for data augmentation, anomaly detection, and applications requiring latent space control.
- **GAN**: Preferred when highest-quality image fidelity is required (super-resolution, image synthesis, deepfakes).

In [ ]:
# ── Summary of all saved files ──────────────────────────────────────────────────
import os

print('=' * 55)
print('   Assignment Completed Successfully!')
print('=' * 55)
print('\nGenerated & Saved Files:')
for f in sorted(os.listdir('generated_images')):
    size = os.path.getsize(f'generated_images/{f}')
    print(f'  ✔  {f}  ({size/1024:.1f} KB)')
print('\nModels trained:')
print('  ✔  VAE  — Variational Autoencoder')
print('  ✔  GAN  — Generative Adversarial Network')
print('\nDataset: MNIST (60,000 training images)')